1. How would you define machine learning?


In [1]:
# Mit MachineLearning kann man Code erstellen, welche aus den Daten lernt.
# Dadurch wird es für die vorhergersehene Aufgabe im Optimalfall besser.

2. Can you name four types of applications where it shines?




In [2]:
# 1. Wenn Probleme bestehen bei denen sich die Sachlage immer wieder ändert.
# 2. Um Muster/Anomalien zu erkennen, welche vorher unbekannt waren, oder nicht klar (Clustering, Dimensionale Reduktion)
# 3. Bilderkennung und -kategorisierung
# 4. Wenn Probleme keine algorithmische Lösung haben



What is a labeled training set?

In [3]:
# Ein labeled training set, ist ein Trainingset mit den bereits gewünschten Zielwert. D.h. der Zielwert ist jeweils klar.
# Zum Beispiel Bilder mit Hunden, die bereits mit dem Zielkategorie "Hund", bestückt wurden.

What are the two most common supervised tasks?


In [4]:
# Einerseits Klassifikation  und andererseits Regression.

Can you name four common unsupervised tasks?

In [5]:
# Clustering, Dimensionale Reduktion, Anomalyerkennung und Neuheitserkennung(Novelty Detection).

What type of algorithm would you use to allow a robot to walk in various unknown terrains?

In [6]:
#Reinforcement learning

What type of algorithm would you use to segment your customers into multiple groups?






In [7]:
#Classification, solange man die Gruppen kennt. Ansonsten Clustering.

Would you frame the problem of spam detection as a supervised learning problem or an unsupervised learning problem?



In [8]:
# Ich würde es als supervised learning problem definieren.

What is an online learning system?





In [9]:
# Im Gegensatz zu Batch-Learning kann das System inkrementiell von den neuen Fällen lernen und somit sein Verhalten anpassen.

What is out-of-core learning?

In [10]:
# Out-of-core learning ist ähnlich wie online-learning jedoch sind die Daten so gross, dass diese nicht in den Arbeitsspeicher von nur einer Masachine passt.
# So wird also immer wieder ein Stück der neuen Daten geladen und der Algorthmus darauf trainiert.

What type of algorithm relies on a similarity measure to make predictions?





In [11]:
# Ähnlichkeiten wird mit instance-based learning trainiert. Mit Hilfe von den Trainingsdaten wird es dann neue Instanzen entsprechend zuordnen.

What is the difference between a model parameter and a model hyperparameter?





In [12]:
# So wie ich das verstehe sind Modelparameter, die Parameter welche das Modell schlussendlich nutzt um Vorhersagen zu treffen.
# Hyperparameter steuern im Gegensatz dazu, wie das Modell selbst lernt.

What do model-based algorithms search for? What is the most common strategy they use to succeed? How do they make predictions?





In [13]:
# Modelbasierte Algorithmen suchen nach Zusammenhängen, statt wie instanzbasierte, die ähnliches suchen.
# Sie prüfen nach einem Algorithmus, wo die Daten hingehören könnten aufgrund eines Wertes und ordnen mit dieser mathematischen Formel dann den entsprechend Wert zu.

Can you name four of the main challenges in machine learning?



In [14]:
# Qualitativ schlechte Daten
# Overfitting
# Underfitting
# Zu wenig repräsentative Daten

If your model performs great on the training data but generalizes poorly to new instances, what is happening? Can you name three possible solutions?



In [15]:
# Overfitting
# Lösungen: 1. Modell vereinfachen 2. Mehr Trainingsdaten sammeln 3. Datenfehler und Ausreisser entfernen

What is a test set, and why would you want to use it?



In [68]:
# Ein Testset wird benutzt, um die Qualität des endgültigen Modells zu prüfen.

What is the purpose of a validation set?



In [17]:
# Ein Validationsset benutzt man, um verschiedene Modelle zu testen. So kann man das beste auswählen.

What is the train-dev set, when do you need it, and how do you use it?



In [18]:
# Train-dev set benutzt man, wenn man viele Daten hat, aber wenig repräsentative.
# Ein Teil der Trainingsdaten wird extra gesondert aufbewahrt, um Overfitting zu verhindern.

What can go wrong if you tune hyperparameters using the test set?



In [67]:
# Overfitting auf das test set.

Code zur Vorbereitung für die Antworten zu Kapitel 2.


In [20]:
import sys
import sklearn
from packaging import version
from pathlib import Path
import pandas as pd
import tarfile
import urllib.request
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV
import numpy as np

In [21]:
def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
    with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path="datasets")
    return pd.read_csv(Path("datasets/housing/housing.csv"))

housing = load_housing_data()

C:\Users\yhaen\AppData\Local\Temp\ipykernel_14156\1655008950.py:8: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  housing_tarball.extractall(path="datasets")


In [22]:
def shuffle_and_split_data(data, test_ratio):
    shuffled_indices = np.random.permutation(len(data))
    test_set_size = int(len(data) * test_ratio)
    test_indices = shuffled_indices[:test_set_size]
    train_indices = shuffled_indices[test_set_size:]
    return data.iloc[train_indices], data.iloc[test_indices]

train_set, test_set = shuffle_and_split_data(housing, 0.2)

np.random.seed(42)

In [23]:
housing["income_cat"] = pd.cut(housing["median_income"],
                               bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                               labels=[1, 2, 3, 4, 5])

In [24]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(housing, test_size=0.2, random_state=42)

In [25]:
from sklearn.model_selection import StratifiedShuffleSplit


strat_train_set, strat_test_set = train_test_split(
    housing, test_size=0.2, stratify=housing["income_cat"], random_state=42)

In [26]:
for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

In [27]:
housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

In [28]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_array, check_is_fitted

class StandardScalerClone(BaseEstimator, TransformerMixin):
    def __init__(self, with_mean=True):  # no *args or **kwargs!
        self.with_mean = with_mean

    def fit(self, X, y=None):  # y is required even though we don't use it
        X = check_array(X)  # checks that X is an array with finite float values
        self.mean_ = X.mean(axis=0)
        self.scale_ = X.std(axis=0)
        self.n_features_in_ = X.shape[1]  # every estimator stores this in fit()
        return self  # always return self!

    def transform(self, X):
        check_is_fitted(self)  # looks for learned attributes (with trailing _)
        X = check_array(X)
        assert self.n_features_in_ == X.shape[1]
        if self.with_mean:
            X = X - self.mean_
        return X / self.scale_

In [29]:
from sklearn.cluster import KMeans
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics.pairwise import rbf_kernel # Import rbf_kernel

class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, n_init=10,
                              random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self  # always return self!

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)

    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

In [30]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LinearRegression
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline
from inspect import Signature, signature, Parameter
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.compose import make_column_selector, make_column_transformer

num_attribs = ["longitude", "latitude", "housing_median_age", "total_rooms",
               "total_bedrooms", "population", "households", "median_income"]
cat_attribs = ["ocean_proximity"]

cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"))

In [31]:


cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1., random_state=42)
similarities = cluster_simil.fit_transform(housing[["latitude", "longitude"]])

def column_ratio(X):
    return X[:, [0]] / X[:, [1]]

def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]  # feature names out

def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler())

log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log, feature_names_out="one-to-one"),
    StandardScaler())
cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1., random_state=42)
default_num_pipeline = make_pipeline(SimpleImputer(strategy="median"),
                                     StandardScaler())
preprocessing = ColumnTransformer([
        ("bedrooms", ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
        ("rooms_per_house", ratio_pipeline(), ["total_rooms", "households"]),
        ("people_per_house", ratio_pipeline(), ["population", "households"]),
        ("log", log_pipeline, ["total_bedrooms", "total_rooms", "population",
                               "households", "median_income"]),
        ("geo", cluster_simil, ["latitude", "longitude"]),
        ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
    ],
    remainder=default_num_pipeline)  # one column remaining: housing_median_age

In [32]:
from sklearn.linear_model import LinearRegression

lin_reg = make_pipeline(preprocessing, LinearRegression())
lin_reg.fit(housing, housing_labels)

,steps,"[('columntransformer', ...), ('linearregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('bedrooms', ...), ('rooms_per_house', ...), ...]"
,remainder,Pipeline(step...ardScaler())])
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [33]:
from sklearn.model_selection import cross_val_score

In [34]:
from sklearn.ensemble import RandomForestRegressor

full_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("random_forest", RandomForestRegressor(random_state=42)),
])
param_grid = [
    {'preprocessing__geo__n_clusters': [5, 8, 10],
     'random_forest__max_features': [4, 6, 8]},
    {'preprocessing__geo__n_clusters': [10, 15],
     'random_forest__max_features': [6, 8, 10]},
]
grid_search = GridSearchCV(full_pipeline, param_grid, cv=3,
                           scoring='neg_root_mean_squared_error', n_jobs=-1) #n_jobs zur nutzung aller kerne
grid_search.fit(housing, housing_labels)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"[{'preprocessing__geo__n_clusters': [5, 8, ...], 'random_forest__max_features': [4, 6, ...]}, {'preprocessing__geo__n_clusters': [10, 15], 'random_forest__max_features': [6, 8, ...]}]"
,scoring,'neg_root_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('bedrooms', ...), ('rooms_per_house', ...), ...]"


Try a support vector machine regressor `(sklearn.svm.SVR)` with various hyperparameters, such as `kernel="linear"` (with various values for the `C` hyperparameter) or `kernel="rbf"` (with various values for the `C` and `gamma` hyperparameters). Note that support vector machines don’t scale well to large datasets, so you should probably train your model on just the first 5,000 instances of the training set and use only 3-fold cross-validation, or else it will take hours. Don’t worry about what the hyperparameters mean for now; we’ll discuss them in Chapter 5. How does the best `SVR` predictor perform?



In [35]:
from sklearn.svm import SVR

full_pipeline_svr = Pipeline([
    ("preprocessing", preprocessing),
    ("svr", SVR())
])

param_grid_svr = [
    {
        'svr__kernel': ['linear'],
        'svr__C': [0.1, 1, 10, 100] #Grobe Wertewahl
    },
    {
        'svr__kernel': ['rbf'],
        'svr__C': [0.1, 1, 10, 100],#Grobe Wertewahl
        'svr__gamma': [0.01, 0.1, 1, 10] #Grobe Wertewahl
    }
]


grid_search_svr = GridSearchCV(full_pipeline_svr, param_grid_svr, cv=3, #3-fache Cross-Validation
                               scoring='neg_root_mean_squared_error', verbose=2, n_jobs=-1)

grid_search_svr.fit(housing.iloc[:5000], housing_labels.iloc[:5000]) #Nur die ersten 5000 Instanzen




Fitting 3 folds for each of 20 candidates, totalling 60 fits


C:\Users\yhaen\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=20.
  warnings.warn(


,estimator,"Pipeline(step...svr', SVR())])"
,param_grid,"[{'svr__C': [0.1, 1, ...], 'svr__kernel': ['linear']}, {'svr__C': [0.1, 1, ...], 'svr__gamma': [0.01, 0.1, ...], 'svr__kernel': ['rbf']}]"
,scoring,'neg_root_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('bedrooms', ...), ('rooms_per_house', ...), ...]"


In [37]:
best_svr_rmse = -grid_search_svr.best_score_
best_svr_rmse

np.float64(79441.04502352017)

In [38]:
best_svr_params = grid_search_svr.best_params_
best_svr_params

{'svr__C': 100, 'svr__kernel': 'linear'}

Try replacing the `GridSearchCV` with a `RandomizedSearchCV`

In [39]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint,uniform, geom, expon,loguniform

full_pipeline_svr = Pipeline([
    ("preprocessing", preprocessing),
    ("svr", SVR())
])

param_randomsearch = [
    {
        'svr__kernel': ['linear'],
        'svr__C': loguniform(0.1,1,10,100)
    },
    {
        'svr__kernel': ['rbf'],
        'svr__C': loguniform(0.1,1,10,100),
        'svr__gamma': expon(scale=1.0)
    }
]


randomized_search = RandomizedSearchCV(full_pipeline_svr, param_randomsearch, cv=3, #3-fache Cross-Validation
                               scoring='neg_root_mean_squared_error', verbose=2, random_state=42 ,n_jobs=-1) # Überall wo Zufall drin ist sollte ein random_state sein. Das Buch nutzt meist 42.

randomized_search.fit(housing.iloc[:5000], housing_labels.iloc[:5000]) #Nur die ersten 5000 Instanzen

Fitting 3 folds for each of 10 candidates, totalling 30 fits


C:\Users\yhaen\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=20.
  warnings.warn(


,estimator,"Pipeline(step...svr', SVR())])"
,param_distributions,"[{'svr__C': <scipy.stats....0020B39D9C6E0>, 'svr__kernel': ['linear']}, {'svr__C': <scipy.stats....0020B38415E50>, 'svr__gamma': <scipy.stats....0020B38416FD0>, 'svr__kernel': ['rbf']}]"
,n_iter,10
,scoring,'neg_root_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [40]:
randomized_search_rmse = -randomized_search.best_score_
randomized_search_rmse

np.float64(81640.66029265171)

In [41]:
randomized_search_params = randomized_search.best_params_
randomized_search_params

{'svr__C': np.float64(72.59548184287536), 'svr__kernel': 'linear'}

Try adding a `SelectFromModel` transformer in the preparation pipeline to select only the most important attributes.

In [42]:
from sklearn.feature_selection import SelectFromModel

help(SelectFromModel)

Help on class SelectFromModel in module sklearn.feature_selection._from_model:

class SelectFromModel(sklearn.base.MetaEstimatorMixin, sklearn.feature_selection._base.SelectorMixin, sklearn.base.BaseEstimator)
 |  SelectFromModel(
 |      estimator,
 |      *,
 |      threshold=None,
 |      prefit=False,
 |      norm_order=1,
 |      max_features=None,
 |      importance_getter='auto'
 |  )
 |
 |  Meta-transformer for selecting features based on importance weights.
 |
 |  .. versionadded:: 0.17
 |
 |  Read more in the :ref:`User Guide <select_from_model>`.
 |
 |  Parameters
 |  ----------
 |  estimator : object
 |      The base estimator from which the transformer is built.
 |      This can be both a fitted (if ``prefit`` is set to True)
 |      or a non-fitted estimator. The estimator should have a
 |      ``feature_importances_`` or ``coef_`` attribute after fitting.
 |      Otherwise, the ``importance_getter`` parameter should be used.
 |
 |  threshold : str or float, default=None


In [43]:
selector_pipeline = Pipeline([
    ('preprocessing', preprocessing),
    ('selector', SelectFromModel(RandomForestRegressor(random_state=42),
                                 threshold=0.005)),  # min feature importance
    ('svr', SVR(
        kernel=randomized_search.best_params_["svr__kernel"], #Bestes Kernel ist linear, Linear hat kein Gamma
        C=randomized_search.best_params_["svr__C"]
                )),
])

selector_pipeline.fit(housing.iloc[:5000], housing_labels.iloc[:5000])

C:\Users\yhaen\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=20.
  warnings.warn(


,steps,"[('preprocessing', ...), ('selector', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('bedrooms', ...), ('rooms_per_house', ...), ...]"
,remainder,Pipeline(step...ardScaler())])
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


Try creating a custom transformer that trains a k-nearest neighbors regressor `(sklearn.neighbors.KNeighborsRegressor)` in its `fit()` method, and outputs the model’s predictions in its `transform()` method. Then add this feature to the preprocessing pipeline, using latitude and longitude as the inputs to this transformer. This will add a feature in the model that corresponds to the housing median price of the nearest districts.

In [44]:
from sklearn.neighbors import KNeighborsRegressor

class NearestNeighborsRegressor(BaseEstimator, TransformerMixin):
    def __init__(self, n_neighbors=5):
      self.n_neighbors = n_neighbors

    def fit(self, X, y):
      self.model_ = KNeighborsRegressor(n_neighbors=self.n_neighbors)
      self.model_.fit(X, y)
      return self

    def transform(self, X):
      preds = self.model_.predict(X)
      return preds.reshape(-1, 1)

    def get_feature_names_out(self, names=None):
        return ["nearest_district_price"]

In [45]:
preprocessing = ColumnTransformer([
        ("bedrooms", ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
        ("rooms_per_house", ratio_pipeline(), ["total_rooms", "households"]),
        ("people_per_house", ratio_pipeline(), ["population", "households"]),
        ("log", log_pipeline, ["total_bedrooms", "total_rooms", "population",
                               "households", "median_income"]),
        ("geo", cluster_simil, ["latitude", "longitude"]),
        ("geo_knn", NearestNeighborsRegressor(n_neighbors=5), ["latitude", "longitude"]),
        ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
    ], remainder=default_num_pipeline)  # one column remaining: housing_median_age

knn_pipeline = Pipeline([
    ('preprocessing', preprocessing),
    ('selector', SelectFromModel(RandomForestRegressor(random_state=42),
                                 threshold=0.005)),
    ('svr', SVR(
        kernel=randomized_search.best_params_["svr__kernel"], #Bestes Kernel ist linear, Linear hat kein Gamma
        C=randomized_search.best_params_["svr__C"]
                ))
])
knn_pipeline

,steps,"[('preprocessing', ...), ('selector', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('bedrooms', ...), ('rooms_per_house', ...), ...]"
,remainder,Pipeline(step...ardScaler())])
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


Automatically explore some preparation options using `RandomizedSearchCV`.

In [46]:
param_distrib_knn_random = {
    'preprocessing__geo__n_clusters': randint(5, 15), #Clusteranzahl prüfen
    'preprocessing__geo__gamma': expon(scale=1.0), #Geo_gamma testen
    'preprocessing__geo_knn__n_neighbors': randint(3, 10), #Prüfen ob eine andere Nachbarszahl besser ist
    'svr__C': loguniform(0.1,1,10,100),
    'svr__kernel': ['linear', 'rbf'], #Prüfen was besser ist. Wenn rbf dann auch gamma
    'svr__gamma': expon(scale=1.0),
    'selector__threshold': uniform(0.001, 0.01) #Den Selectortreshhold verändern

}


randomized_pre_search = RandomizedSearchCV(knn_pipeline, param_distributions=param_distrib_knn_random, cv=2, #2-fache Cross-Validation - Dauert sonst zu lange..,
                                           scoring="neg_root_mean_squared_error", random_state=42,# RandomizedSearch nutzt Zufall = random_state zur Replikation
                                           verbose=2, n_jobs=-1)

randomized_pre_search.fit(housing.iloc[:5000], housing_labels.iloc[:5000])


# Mit dem Weglassen von einigen Parametern könnte die Zeit weiter reduziert werden. Lokal auf JupyterNotes dauerte es ca. 5 Minuten (I9-14900F Prozessor, 32GB RAM)
# Google Colab mehr als 30 Minuten.

Fitting 2 folds for each of 10 candidates, totalling 20 fits


C:\Users\yhaen\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=20.
  warnings.warn(


,estimator,Pipeline(step...l='linear'))])
,param_distributions,"{'preprocessing__geo__gamma': <scipy.stats....0020B39DB09D0>, 'preprocessing__geo__n_clusters': <scipy.stats....0020B39D9DA90>, 'preprocessing__geo_knn__n_neighbors': <scipy.stats....0020B3857B890>, 'selector__threshold': <scipy.stats....0020B35A11260>, ...}"
,n_iter,10
,scoring,'neg_root_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,2
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [47]:
randomized_pre_search.best_score_

np.float64(-119731.6158750908)

In [48]:
randomized_pre_search.best_params_

{'preprocessing__geo__gamma': np.float64(0.7439661879833189),
 'preprocessing__geo__n_clusters': 14,
 'preprocessing__geo_knn__n_neighbors': 6,
 'selector__threshold': np.float64(0.010737555188414592),
 'svr__C': np.float64(27.091152151232272),
 'svr__gamma': np.float64(0.09497731319666444),
 'svr__kernel': 'rbf'}

Try to implement the StandardScalerClone class again from scratch, then add support for the `inverse_transform()` method: executing scaler.`​inverse_transform(scaler.fit_transform(X))` should return an array very close to `X`. Then add support for feature names: set `feature_names_in_` in the `fit()` method if the input is a DataFrame. This attribute should be a NumPy array of column names. Lastly, implement the `get_feature_names_out()` method: it should have one optional `input_features=None` argument. If passed, the method should check that its length matches `n_features_in_`, and it should match `feature_names_in_` if it is defined; then `input_features` should be returned. If `input_features` is None, then the method should either return `feature_names_in_` if it is defined or `np.array(["x0", "x1", ...])` with length `n_features_in_` otherwise.



In [65]:
# Selbst erarbeitet, mit Unterstützung von ChatGPT zum Verständnis

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_array, check_is_fitted
from sklearn.utils.validation import validate_data
import numpy as np
import pandas as pd

class StandardScalerClone(TransformerMixin, BaseEstimator):
    def __init__(self, with_mean=True):  # no *args or **kwargs!
        self.with_mean = with_mean

    def fit(self, X, y=None):
        if hasattr(self, "feature_names_in_"): #Falls ein neuer Fit geschieht und es bereits das Attribut hat, muss das gelöscht werden (evtl. ist der neue Datensatz ein np.array statt dataframe)
            del self.feature_names_in_

        if isinstance(X, pd.DataFrame): # Falls X ein Dataframe ist müssen wir die Spaltentitel speichern, um sie in get_feature_names_out() wieder auszugeben.
            self.feature_names_in_ = np.array(X.columns)

        X = check_array(X)  # Prüfer von sklearn, ob es ein passendes Array ist
        self.mean_ = X.mean(axis=0) # Mittelwert pro Spalte
        self.scale_ = X.std(axis=0) # Standardabweichung pro Spalte
        self.n_features_in_ = X.shape[1]  #Anzahl Spalten
        return self

    def transform(self, X):
        check_is_fitted(self)  # Prüfen ob fit() bereits ausgeführt wurde
        X = check_array(X)     # Erneute Prüfung, ob es ein gültiges Array ist
        X = validate_data(self, X, ensure_2d=True, reset=False) # Von Scikit verlangt um check_estimator zu bestehen, Assert gibt Fehlermeldung
        if self.with_mean:
            X = X - self.mean_ # Mittelwert abziehen
        return X / self.scale_ # Durch Standardabweichung teilen

    def inverse_transform(self, X): # Zurück transformieren
        check_is_fitted(self) # Prüfen ob fit() bereits ausgeführt wurde
        X = check_array(X) # Erneute Prüfung, ob es ein gültiges Array ist
        X = validate_data(self, X, ensure_2d=True, reset=False) # Von Scikit verlangt um check_estimator zu bestehen, Assert gibt Fehlermeldung
        if self.with_mean: # Falls mit Mittelwert
            X = X * self.scale_ # Zurückrechnen --> Standardskalierung multiplizieren und Mittelwert addieren
            X = X + self.mean_
        else: # Falls ohne Mittelwert
            X = X * self.scale_ # Zurückrechnen --> Nur Standardskalierung multiplizieren
        return X

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self) #Prüfen ob fit() ausgeführt wurde
        if input_features is None: # Falls input_features nicht gesetzt wurde, prüfen ob feature_names_in und zurückgeben, sonst Standardnamen
          if hasattr(self, "feature_names_in_"): # Nur zurückgeben, wenn es wirklich das Attribut hat
            return self.feature_names_in_
          else:
            return np.array(["x"+str(i) for i in range(self.n_features_in_)]) # Liste erstellen mit der Anzahl Spalten und in np.array packen --> x0, x1, x2 ...
        else:
          input_features = np.array(input_features)
          if len(input_features) != self.n_features_in_: #Prüfen ob die Anzahl der Spalten passt
              raise ValueError("Anzahl Spalten stimmt nicht überein")
          if hasattr(self, "feature_names_in_"):
            if not np.array_equal(input_features, self.feature_names_in_): #Prüfen ob die Spaltennamen stimmen:
              raise ValueError("input_features stimmen nicht mit feature_names_in_ überein") 
          return input_features


In [66]:
from sklearn.utils.estimator_checks import check_estimator

check_estimator(StandardScalerClone())

[{'estimator': StandardScalerClone(),
  'check_name': 'check_estimator_cloneable',
  'exception': None,
  'status': 'passed',
  'expected_to_fail': False,
  'expected_to_fail_reason': 'Check is not expected to fail'},
 {'estimator': StandardScalerClone(),
  'check_name': 'check_estimator_cloneable',
  'exception': None,
  'status': 'passed',
  'expected_to_fail': False,
  'expected_to_fail_reason': 'Check is not expected to fail'},
 {'estimator': StandardScalerClone(),
  'check_name': 'check_estimator_tags_renamed',
  'exception': None,
  'status': 'passed',
  'expected_to_fail': False,
  'expected_to_fail_reason': 'Check is not expected to fail'},
 {'estimator': StandardScalerClone(),
  'check_name': 'check_valid_tag_types',
  'exception': None,
  'status': 'passed',
  'expected_to_fail': False,
  'expected_to_fail_reason': 'Check is not expected to fail'},
 {'estimator': StandardScalerClone(),
  'check_name': 'check_estimator_repr',
  'exception': None,
  'status': 'passed',
  'expec